### Issuer
### Acquirer
### Refund
### Chargeback

#

### **Full payment ecosystem:**
Cardholder (Customer)

Issuer (Chase, Citi, BofA)

Card Network (Visa, Mastercard)

Acquirer / Sponsor Bank (SVB, Well Fargo, Worldpay)

PayFac (Strip, Square, Toast, Mindbody)

Sub-merchant (food truck, dentist, gym, freelancer)

cardholder (Customer)

### **There are two loops running simultaneously**
- #### The Authorization Loop (happens in ~1second)
The issuer checks: does this cardholder have funds? Does this look fraudulent? They respond in real time. Noboday moves money yet - this is just a yes/no decision.

Sub-merchant -> PayFac -> Acquirer -> Card Network -> Issuer -> Approve/Decline

Sub-merchant <- Payfac <- Acquirer <- Card Network <-

- #### The Settlement Loop (happens in 1-2 days)
Issuer -> Card Network -> Acquirer -> PayFac -> Sub-merchant.

Actual money moves. The issuer debits the cardholder's account. Funds flow down the chain, with each layer taking a small cut, until the sub-merchant receives their payout minus fees.

#

#

### **Interchange fees**
The Full Fee Flow on a $100 Transaction

Customer pays merchant:                        $100.00

Interchange (~1.5–2.5%) → Issuer:              $2.00

Assessment (~0.13-0.15%)     → Card Network:         $0.13

Processing margin (~0.2-0.5%)       → Acquirer/Stripe:      $0.27


Merchant receives (MDR - Merchant Discount Rate):                       $97.60

- The acquirer collects the full MDR from the merchant and then passes the interchange portion upstream to the issuer via the card netwrok.

**Interchange Is Not One Rate — It's a Table of Hundreds**

This is where it gets complex. Visa and Mastercard publish interchange rate tables with hundreds of different rates depending on:

**Card type**

Debit card: ~0.05% + $0.22 (much lower — no credit risk, no rewards)
Basic credit card: ~1.5%
Rewards credit card: ~2.0%
Premium/Infinite card: ~2.5%+
Corporate card: ~2.7%+

**Merchant category (MCC)**

Grocery stores: lower interchange (high volume, thin margins, political pressure)
Airlines: moderate
Restaurants: moderate
General retail: standard
Digital goods: higher
Financial services: higher

**Transaction type**

Card present (chip/tap): lower — less fraud risk
Card not present (online): higher — more fraud risk
Tokenized transaction: slightly lower — more secure

**Geography**

Domestic transactions: lower
Cross-border transactions: higher — additional currency and compliance risk

#

#

**Where Each Party's ML Problems Live**

This is the most important frame for your Stripe prep — every party in this chain has distinct ML challenges:
Cardholder / Customer

Are they who they claim to be? (identity fraud)
Is this their normal behavior? (account takeover detection)
Will they dispute this charge later? (friendly fraud prediction)

**Issuer**

Should I approve or decline this transaction? (authorization fraud model)
Is this account compromised? (ATO detection)
Will this cardholder default? (credit risk)
Am I declining too many legitimate transactions? (false positive optimization)

**Card Network**

Network-level fraud pattern detection across all issuers and acquirers
Interchange fee optimization
Compliance monitoring

**Acquirer / Sponsor Bank**

Is this PayFac managing their sub-merchants responsibly?
Is overall portfolio chargeback rate within network thresholds?
Which PayFacs are becoming systemic risk?

**PayFac (Stripe)**

Should I onboard this sub-merchant? (underwriting ML)
Is this sub-merchant behaving normally? (monitoring ML)
Is this specific transaction fraudulent? (Radar)
Will this issuer approve this transaction? (authorization optimization)
Is this a fraudulent card being used across my sub-merchants? (network fraud signals)
Is this cardholder who they say they are? (Stripe Identity)

**Sub-merchant**

Largely a consumer of Stripe's ML outputs, not a builder

#

## Acquirer

covering all six functions:

- **Merchant underwriting**—  evaluating merchants before onboarding

- **Transaction routing** — sending auth requests to the card network and back

- **Settlement** — moving actual money to the merchant's account

- **Chargeback management** — handling disputes and representment

- **Compliance and monitoring** — keeping the portfolio within card network rules

- **Interchange and fee management** — collecting MDR, passing interchange to issuer

### **Issuer**
- **The Issuer (or Issuing bank)** is the bank that gave the cardholder their credit or debit card.
- **The Five parties in Every Card Transaciton**
    - Cardholder -> Issuer -> Card Network -> Acquirer -> Merchant
    - (You)  -> (Chase) -> (Visa/MC) -> (Stripe) -> (Amazon)
- **Cardholder** - person spending money
- **Issuer** - cardholder's banck (Chase, Citi, Bank of America, etc.)
- **Card Network** - the rails (Visa, Mastercard, Amex)
- **Acquirer** - merchant's bank/payment processor (Stripe sits here)
- **Merchant** - business receiving payment



**Why Issuers Matter for ML at Stripe**
- This is where it gets directly relevant to your target role. The Stripe JD specifically mentions "issuers" as one of the key payment entities their models are built around. Here's why:
- **Authorization decisions live with the issuer.** When a transaction is submitted, the issuer approves or declines it in real time — in under a second. Their decision is based on their own fraud models. Stripe has no control over this decision, but they can influence it.
- **Authorization rate optimization is one of Stripe's most valuable ML problems.** If an issuer is incorrectly declining legitimate transactions (false positives on their fraud model), Stripe loses revenue and merchants lose sales. Stripe's ML team builds models to predict which transactions are likely to be declined and either retry them differently, route them through a different network, or send additional signals to the issuer to improve approval odds. This is what "authorization optimization" in the JD refers to.
- **Issuer behavior is a feature.** Different issuers have very different decline rates, fraud tolerance, and behavior patterns. A transaction declined by a conservative issuer means something different than one declined by a permissive issuer. Issuer identity and historical behavior are powerful features in fraud and authorization models.
The issuer sees different data than Stripe. The issuer knows your full account history, balance, spending patterns. Stripe only sees the transaction. This information asymmetry is a core challenge in payments ML — you're making fraud decisions with incomplete information, which is why the models have to be very good.

- In short: the issuer is the gatekeeper of every transaction, and a significant portion of Stripe's ML work is either predicting issuer behavior or influencing it. Knowing this cold will come across very well in an interview. Sonnet 4.6

### Refund:
- **A refund is a merchant**-initiated return of funds. The merchant agrees the money should to back - customer complained, item returned, whatever. The merchant controls it.
- **Who Initiates**: Merhcant
- **Merchant consent**: Yes
- **Timeline**: Days
- **Cost to merchant**: Transaction amount only
- **Dispute process**: None
- **Signal for fraud**: Weak
- **Reversible**: Sometimes

### Chargeback:
- **A chargeback is a cardholder**-initiated dispute filed directly with their bank, bypassing the merchant entirely. The bank forcibly reverses the transaction. The merchant doesn't agree - they fight it or absorb the loss.
- **Who Initiates**: Cardholder via issuing bank
- **Merchant consent**: No
- **Timeline**: 60-120 days after transaction
- **Cost to merchant**: Transacton amount + chargeback fee ($15 - $100) + potential penalty
- **Dispute process**: representment process
- **Signal for fraud**: Strong
- **Reversible**: Very hard

**How the Full Chargeback Cycle Works**

- Cardholder disputes a charge with their bank
- Issuing bank provisionally reverses the funds and notifies Stripe (as the acquirer)
- Stripe notifies the merchant and asks: do you want to fight this?
- Merchant submits representment — evidence package showing the transaction was valid (delivery confirmation, signed receipt, customer communications, IP logs, etc.)
- Issuing bank reviews and makes a final decision — either upholds the chargeback or reverses it back in the merchant's favor
- If the merchant loses, they can escalate to arbitration (Visa/Mastercard makes the final call) — expensive and rarely worth it

**Why It Matters for ML**
Representment data is actually a valuable ML signal that most people overlook:
**Win/loss patterns reveal fraud type.** If a merchant consistently wins representments, those chargebacks were likely friendly fraud (legitimate transactions disputed dishonestly). If they consistently lose, the transactions were probably genuinely fraudulent. This win/loss ratio is a feature you could engineer for merchant risk scoring.
**Representment rate as a merchant signal.** A merchant who never fights chargebacks might be a drop-shipper or high-risk operator who knows the goods weren't delivered. A merchant who fights everything and wins most of the time is a strong legitimacy signal.
**Label refinement.** Raw chargeback = fraud label. But if the merchant won representment, that label was wrong. Incorporating representment outcomes into your training labels produces cleaner ground truth — a meaningful model quality improvement that you could propose as an ML project.